# Mission Veteran AI – Veteran Career Navigator Agent
## Skill Mapping, Gap Analysis & SkillsBuild Recommendations
**IBM SkillsBuild AI Lab 2026 · U.S. 2026 Fall Cohort**

---

### What this notebook does
1. Installs required dependencies (pandas, openpyxl, chromadb, langchain-ibm).
2. Loads the canonical skill and career-path data from `data/FoundationG3Project.xlsx`.
3. Loads the IBM SkillsBuild course catalog from `knowledge-bases/skillsbridge_courses.json`.
4. Builds an in-memory ChromaDB vector store (RAG corpus) from both sources.
5. Tests `map_military_skills()` — translates an occupational specialty to a civilian skill inventory.
6. Tests `analyze_skills_gap()` — identifies gaps and retrieves SkillsBuild course recommendations.
7. Runs edge-case and boundary tests.

### Input → Output
| Input | Output |
|---|---|
| Service branch + specialty code + PII-free service summary + target career path | Civilian skill inventory → gap report → SkillsBuild course recommendations |

### Before you run
- Replace `YOUR_API_KEY_HERE` in Cell 2 with your watsonx.ai API key.
- Confirm your project ID in Cell 2.
- Ensure `data/FoundationG3Project.xlsx` and `knowledge-bases/skillsbridge_courses.json` are present.
- Run cells top to bottom in order.

> **Responsible AI notice:** This notebook generates draft content only. It does not make hiring,
> eligibility, benefits, legal, medical, or financial decisions. All SkillsBuild course recommendations
> are suggestions only — verify current availability at skillsbuild.org before enrolling.

---
## Cell 1 · Install dependencies

In [ ]:
%pip install --quiet \
    ibm-watsonx-ai>=1.1.0 \
    python-dotenv>=1.0.0 \
    openpyxl>=3.1.0 \
    pandas>=2.0.0 \
    chromadb>=0.5.0 \
    langchain-ibm>=0.1.0 \
    langchain>=0.2.0
print('✅  Dependencies installed.')

---
## Cell 2 · Configure credentials

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()  # loads .env if present locally

# Replace these if not using a .env file
os.environ.setdefault('WATSONX_API_KEY',    '')
os.environ.setdefault('WATSONX_PROJECT_ID', '')
os.environ.setdefault('WATSONX_URL',        'https://us-south.ml.cloud.ibm.com')

assert os.environ['WATSONX_API_KEY'] != 'YOUR_API_KEY_HERE', \
    'Set WATSONX_API_KEY before continuing.'
print('✅  Credentials configured.')

---
## Cell 3 · Load Skill Dictionary and Career Paths from FoundationG3Project.xlsx
Handles two environments automatically:
- **watsonx.ai notebook**: reads from project assets. Upload the file first via
  **Assets → New asset → Data → Local file** and select `FoundationG3Project.xlsx`.
- **Local**: reads from `../data/FoundationG3Project.xlsx`.

In [ ]:
import os
import pandas as pd

XLSX_FILENAME = 'FoundationG3Project.xlsx'
LOCAL_PATH    = '../data/FoundationG3Project.xlsx'
TMP_PATH      = f'/tmp/{XLSX_FILENAME}'

def _resolve_xlsx() -> str:
    """Return a local filesystem path to the xlsx, downloading from project assets if needed."""
    # ── watsonx.ai notebook environment ──────────────────────────────────────
    if os.environ.get('RUNTIME_ENV_APSX_URL'):
        from ibm_watsonx_ai import APIClient, Credentials
        client = APIClient(Credentials(
            url=os.environ['RUNTIME_ENV_APSX_URL'],
            token=os.environ['USER_ACCESS_TOKEN'],
        ))
        assets = client.data_assets.get_details()['resources']
        matches = [a for a in assets if a['metadata']['name'] == XLSX_FILENAME]
        if not matches:
            raise FileNotFoundError(
                f"'{XLSX_FILENAME}' not found in project assets. "
                "Upload it via Assets → New asset → Data → Local file."
            )
        asset_id = matches[0]['metadata']['asset_id']
        client.data_assets.download(asset_id, filename=TMP_PATH)
        print(f'Downloaded {XLSX_FILENAME} from project assets → {TMP_PATH}')
        return TMP_PATH

    # ── Local environment ─────────────────────────────────────────────────────
    if os.path.exists(LOCAL_PATH):
        print(f'Using local file: {LOCAL_PATH}')
        return LOCAL_PATH

    raise FileNotFoundError(
        f"Cannot find '{XLSX_FILENAME}'. "
        "Locally: ensure data/FoundationG3Project.xlsx exists. "
        "On watsonx.ai: upload via Assets → New asset → Data."
    )

XLSX_PATH = _resolve_xlsx()

df_skills = pd.read_excel(XLSX_PATH, sheet_name='Skill_Dictionary')
df_career = pd.read_excel(XLSX_PATH, sheet_name='Career Paths')

print(f'Skill_Dictionary: {len(df_skills)} rows, columns: {list(df_skills.columns)}')
print(f'Career Paths:     {len(df_career)} rows, columns: {list(df_career.columns)}')
df_career.head(8)

---
## Cell 4 · Load SkillsBuild course catalog

In [ ]:
import json

KB_PATH = '../knowledge-bases/skillsbridge_courses.json'

with open(KB_PATH, 'r') as f:
    courses = json.load(f)

print(f'Loaded {len(courses)} SkillsBuild courses.')
for c in courses[:3]:
    print(f"  {c['course_id']}: {c['title']} ({c['credential']}, {c['estimated_hours']}h)")

---
## Cell 5 · Build ChromaDB RAG vector store
Creates an in-memory vector store from the skill dictionary and course catalog.
In production this would be persisted to watsonx.data / Milvus.

In [ ]:
import chromadb

client_db = chromadb.Client()  # in-memory

# -- Skills collection --
skills_col = client_db.create_collection('skills_dictionary')
skill_docs = [
    f"{row['skill_name']}: {row['civilian_equivalents']}. Synonyms: {row['synonyms']}."
    for _, row in df_skills.iterrows()
    if pd.notna(row.get('skill_name')) and pd.notna(row.get('civilian_equivalents'))
]
skill_ids = [f'SKILL-{i:03d}' for i in range(len(skill_docs))]
skills_col.add(documents=skill_docs, ids=skill_ids)
print(f'Skills collection: {len(skill_docs)} documents indexed.')

# -- Courses collection --
courses_col = client_db.create_collection('skillsbridge_courses')
course_docs = [
    f"{c['title']}: {c['description']} Skills: {', '.join(c['skill_tags'])}."
    for c in courses
]
course_ids = [c['course_id'] for c in courses]
courses_col.add(documents=course_docs, ids=course_ids)
print(f'Courses collection: {len(course_docs)} documents indexed.')

---
## Cell 6 · Test map_military_skills() — Army 92A (Automated Logistical Specialist)

In [ ]:
import sys
sys.path.insert(0, '..')

from toolkits.skill_mapper.map_military_skills import map_military_skills

result = map_military_skills(
    service_branch='Army',
    specialty_code='92A',
    service_summary=(
        'Led a 12-person logistics team supporting time-sensitive supply operations '
        'across multiple sites. Managed $4.2M in equipment and improved on-time '
        'delivery from 82% to 96%. Redesigned the maintenance scheduling process '
        'to remove workflow bottlenecks, reducing backlog 38% in 90 days.'
    ),
    target_career_path='Logistics',
)

if result['error']:
    print(f'ERROR: {result["error"]}')
elif result['pii_warning']:
    print(f'\u26a0  PII WARNING: {result["pii_warning"]}')
else:
    print(f'\u2705  {result["skill_count"]} skills mapped:\n')
    for s in result['skills']:
        print(f"  {s['skill']} ({s['category']})") 


---
## Cell 7 · Test analyze_skills_gap() — Project Management path

In [ ]:
import json as _json
from toolkits.gap_analyzer.analyze_skills_gap import analyze_skills_gap

# Use the skills from Cell 6 - swap to 'Project Management' path
skills_json = _json.dumps(result['skills'])

gap_result = analyze_skills_gap(
    veteran_skills_json=skills_json,
    target_career_path='Project Management',
    job_description='',
)

if gap_result['error']:
    print(f'ERROR: {gap_result["error"]}')
else:
    print(f'Coverage: {gap_result["coverage_pct"]}%\n')

    print('COVERED SKILLS:')
    for s in gap_result['covered_skills']:
        print(f'  \u2713 {s}')

    print('\nSKILL GAPS + SKILLSBUILD RECOMMENDATIONS:')
    for g in gap_result['gap_skills']:
        print(f'  \u2717 {g}')

    print('\nSkillsBuild Recommendations:')
    for course in gap_result.get('skillsbuild_recommendations', []):
        print(f"  \u2192 {course['course_title']} | {course['url']}")

    print(f'\nDisclaimer: {gap_result["disclaimer"]}')


---
## Cell 8 · Edge case tests

In [ ]:
print('=== Edge Case 1: Empty service_summary ===')
r = map_military_skills(
    service_branch='Navy', specialty_code='BM', service_summary=''
)
assert r['error'], 'Expected error for empty service_summary'
print(f'  \u2705  Error returned: {r["error"]}')

print('\n=== Edge Case 2: PII detection - SSN pattern ===')
r3 = map_military_skills(
    service_branch='Army',
    specialty_code='11B',
    service_summary='SSN 123-45-6789 led a team of 10 personnel.',
)
assert r3['pii_warning'], 'Expected PII warning for SSN pattern'
print(f'  \u2705  PII warning returned \u2713')

print('\n=== Edge Case 3: Invalid veteran_skills_json ===')
r4 = analyze_skills_gap(
    veteran_skills_json='not valid json',
    target_career_path='Operations',
    job_description='',
)
assert r4['error'], 'Expected error for invalid JSON'
print(f'  \u2705  Error returned: {r4["error"]}')

print('\n\u2705  All edge case tests passed.')


---
## Cell 9 · Validate Orchestrate tool decorator registration

In [ ]:
# Validate that toolkit tool functions are importable and callable
from toolkits.skill_mapper.map_military_skills import map_military_skills as _sm
from toolkits.gap_analyzer.analyze_skills_gap import analyze_skills_gap as _ga

import inspect
for fn, label in [(_sm, 'map_military_skills'), (_ga, 'analyze_skills_gap')]:
    assert callable(fn), f'{label} is not callable'
    sig = inspect.signature(fn)
    print(f'\u2705  {label}: params={list(sig.parameters.keys())}')

print('\n\u2705  Both toolkit functions importable and callable.')


---
## Next steps

### Import tools and agent into watsonx Orchestrate
```bash
orchestrate toolkits import -f toolkits/skill_mapper/toolkit.yaml --app-id watsonx_credentials
orchestrate toolkits import -f toolkits/gap_analyzer/toolkit.yaml --app-id watsonx_credentials
orchestrate agents import -f agents/skill_mapper_agent.yaml
orchestrate agents import -f agents/gap_analyzer_agent.yaml
```

### Career pipeline preview
The career pipeline adds:
- Full job description ingestion and JD-to-skill keyword extraction.
- Multi-bullet accomplishment intake (all records from `tblRecords`).
- STAR story builder (Situation, Task, Action, Result) for interview prep.
- PDF/document ingestion from TAP curriculum sources.

> **Responsible AI notice:** All output is draft content only. Review before use.
> SkillsBuild course recommendations are suggestions — verify at skillsbuild.org.